# Investigating different reduction function family candidates

10,000 startpoints
10,000 reduction function indexes

## Imports

In [541]:
from hashlib import sha256
import mmh3
from math import e
import random
import numpy as np
import time
import pandas as pd
from tqdm import tqdm

## Constants

In [542]:
N = 2 ** 16  # keyspace
p = 1 - e ** -2  # our table coverage - 86%
t = 80  # number of columns

## Helper functions

In [543]:
# Hash function
def H(x):
    return int.from_bytes(sha256(x.to_bytes(8)).digest())

def H_c(x):
    return sha256(x.to_bytes(8)).digest()  # return bytes directly
    
# mmh reduction function
def r_mmh(N, t, y, i, ell=0):
    # seed = int(i) + (ell*t)
    # seed = i + (ell*t)
    # return mmh3.hash(y, seed, signed=False) % N
    return mmh3.hash(y, i + (ell*t), signed=False) % N

def r(N, t, y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	# return (y + i + ell*t) % N
    # if everything is bytes

    return (y + i + ell*t) & (N - 1) # N is a power of 2 so can use bitwise AND instead of modulo

## Set up

### Build hash table

In [544]:
# choose random unique 100,000 inputs and hash them
points = random.sample(range(N), 10000)
    
# record how long hashing 100000 points takes


hash_table = []
start_time = time.perf_counter()
for i in points:
    hash_table.append(H(i))

hash_time = time.perf_counter() - start_time
print(f"Hashing 10,000 points took {hash_time:.6f} seconds.")

hash_bytes = []
start_time = time.perf_counter()
for i in points:
    hash_bytes.append(H_c(i))
hash_time = time.perf_counter() - start_time
print(f"Hashing 10,000 points took {hash_time:.6f} seconds.")

Hashing 10,000 points took 0.016401 seconds.
Hashing 10,000 points took 0.011786 seconds.


### Select K

- Need 10,000 random 32 bit integers 

In [545]:
K_np = np.array(random.sample(range(2 ** 32), 10000), dtype=np.dtype('>u4'))
# K = np.array(random.sample(range(10000), 10000), dtype=np.uint32)
# K = random.sample(range(2 ** 32), 10000)
# convert all of K to int
K = K_np.astype(int).tolist()

### Table to store results

- we are testing a bunch of indexes, and for the same startpoints used how many are left after reducing

K | mmh3.hash | mmh3.hash64 | mmh3.x64  (recording # points left)

- also want to store how fast each one is

K | mmh3.hash | mmh3.hash64 | mmh3.x64   (recording time taken)

- add columns as we compute

In [546]:
points_data = pd.DataFrame({"K" : K})
time_data = pd.DataFrame({"K" : K})

## Test reduction functions

In [547]:
## normal reduction function
# list to be become column in df
# mmh3_times = []
# mmh3_points = []
# # for each index in K
# for index in tqdm(K):
#     table = set()
#     # reduce all the startpoints and see how many are left with this index
#     start_time = time.perf_counter()
#     for hash in hash_table:
#         point = r(N, t, hash, index)
#         table.add(point)
#     elapsed = time.perf_counter() - start_time
#     mmh3_times.append(elapsed)
#     mmh3_points.append(len(table))

# # add to dataframe
# time_data["normal"] = mmh3_times
# points_data["normal"] = mmh3_points
    
mmh3_times = []
mmh3_points = []
pbar = tqdm(K, smoothing=0)
# for each index in K
for index in pbar:
    table = set()
    # reduce all the startpoints and see how many are left with this index
    start_time = time.perf_counter()
    for hash in hash_table:
        point = r(N, t, hash, index)
        table.add(point)
    elapsed = time.perf_counter() - start_time
    mmh3_times.append(elapsed)
    mmh3_points.append(len(table))

# Extract values from format_dict
stats = pbar.format_dict
total_iterations = stats['n']
total_time = stats['elapsed']

# Prevent division by zero if the loop was empty
if total_time > 0:
    avg_its_per_sec = total_iterations / total_time
    print(f"Overall average speed: {avg_its_per_sec:.2f} it/s")
else:
    print("Loop finished too quickly to measure time.")

# add to dataframe
time_data["normal"] = mmh3_times
points_data["normal"] = mmh3_points
    


## mmh3
# list to be become column in df
mmh3_times = []
mmh3_points = []
pbar = tqdm(K, smoothing=0)
# for each index in K
for index in pbar:
    table = set()
    # reduce all the startpoints and see how many are left with this index
    start_time = time.perf_counter()
    for hash in hash_bytes:
        table.add(r_mmh(N, t, hash, index))
    elapsed = time.perf_counter() - start_time
    mmh3_times.append(elapsed)
    mmh3_points.append(len(table))

# Extract values from format_dict
stats = pbar.format_dict
total_iterations = stats['n']
total_time = stats['elapsed']

# Prevent division by zero if the loop was empty
if total_time > 0:
    avg_its_per_sec = total_iterations / total_time
    print(f"Overall average speed: {avg_its_per_sec:.2f} it/s")
else:
    print("Loop finished too quickly to measure time.")

# add to dataframe
time_data["hash"] = mmh3_times
points_data["hash"] = mmh3_points


100%|██████████| 10000/10000 [00:44<00:00, 223.23it/s]


Overall average speed: 223.23 it/s


100%|██████████| 10000/10000 [00:45<00:00, 221.19it/s]

Overall average speed: 221.18 it/s
